# Georreferenciación

Se incorpora la dimensión espacial al análisis de los comparendos electrónicos, ubicando geográficamente los puntos de control (cámaras) y visualizando su distribución en el territorio. Este análisis permite identificar la cobertura espacial del sistema de vigilancia, detectar zonas con mayor concentración de cámaras y establecer relaciones entre la ubicación de los puntos de control y el volumen de infracciones detectadas, sentando las bases para futuros análisis de dependencia espacial o clusters de infracciones.

## Carga de Librerías

Se importan las librerías necesarias para el análisis geoespacial de los comparendos electrónicos, con el objetivo de ubicar geográficamente las cámaras de vigilancia y visualizar su distribución en el territorio. Se emplean `folium` para la creación de mapas interactivos, `geopandas` para el manejo de datos geoespaciales y `shapely.geometry.Point` para la representación de coordenadas como puntos geográficos. Adicionalmente, se utilizan `scipy.spatial.cKDTree` para la búsqueda espacial eficiente, `branca.colormap` para la generación de escalas de color personalizadas, y `plotly.express` junto con `plotly.graph_objects` para visualizaciones complementarias. Se suprimen los mensajes de advertencia para mantener una salida limpia y centrada en los resultados del análisis geoespacial.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import folium
import geopandas as gpd
import branca.colormap as cm
import plotly.io as pio

from folium import plugins
from shapely.geometry import Point
from scipy.spatial import cKDTree

In [2]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [3]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [4]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [5]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [6]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [7]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

## Geolocalización de las Cámaras

Se realiza la geolocalización de las cámaras de vigilancia presentes en la base de datos, asignando coordenadas de latitud y longitud a cada punto de control para su posterior visualización en mapas interactivos. Para ello, se excluyeron las cámaras de "Mal Parqueo" por tratarse de unidades móviles sin ubicación fija, así como aquellas cuyo nombre incluía especificaciones de sentido de circulación (Norte-Sur, Sur-Norte, etc.), ya que dicha información no brindaba claridad para ubicarlas geográficamente en un punto único. Cada dirección fue contrastada con el dataset de "Fotodetección en Barranquilla" suministrado por Alcaldía de Barranquilla en Datos Abiertos Colombia, el cual proporciona latitud y longitud para cada cámara, así como los tipos de infracción que detecta, permitiendo verificar que correspondan al mismo punto de control. Las cámaras no encontradas en este dataset fueron geolocalizadas manualmente mediante Google Maps. Cabe mencionar que se intentó automatizar este proceso de búsqueda de coordenadas, pero la forma en que están escritas las direcciones (con inconsistencias en la nomenclatura, abreviaciones y falta de estandarización) no posibilita esta automatización, requiriendo una validación manual caso por caso.

**Fuente**: https://www.datos.gov.co/Transporte/Fotodetecci-n-en-Barranquilla/cpp6-je64/about_data

In [8]:
direcciones = [
    "VIA 11 CON CARRERA 8",
    "CARRERA 51B CON CALLE 79",
    "CALLE 30 CON CARRERA 6B",
    "CARRERA 53 ENTRE CALLE 104 Y 106",
    "CALLE 45 CON CARRERA 1",
    "CALLE 30 CON CARRERA 8",
    "CALLE 82 CON CARRERA 51B",
    "CARRERA 51B CON CALLE 103",
    "CARRERA 6 CON CALLE 72",
    "AVENIDA CIRCUNVALAR CON CARRERA 9G",
    "VIA 40 CON CALLE 73",
    "CALLE 45 CON CARRERA 20",
    "CALLE 82 CON CARRERA 56",
    "CALLE 45B CON CARRERA 14",
    "CALLE 45 CON CARRERA 21",
    "CARRERA 15 CON CALLE 21",
    "CALLE 72 CON CARRERA 44",
    "CALLE 84 ENTRE CARRERA 59 Y 59B",
    "CALLE 94 CON CARRERA 58",
    "CARRERA 46 CON CALLE 100",
    "CALLE 56 CON CARRERA 14",
    "CALLE 19 CON CARRERA 4C",
    "CALLE 53 CON CARRERA 45",
    "CALLE 87 CON CARRERA 21",
    "CALLE 76 CON CARRERA 38C-100",
    "CALLE 19 CON CARRERA 3D",
    "CARRERA 53 CON CALLE 86",
    "CALLE 70 CON CARRERA 46",
    "CALLE 34 CON CARRERA 45",
    "AVENIDA CIRCUNVALAR CON CARRERA 31",
    "CALLE 98 CON CARRERA 56",
    "CALLE 61 CON CARRERA 35",
    "CARRERA 27 CON CALLE 82C"
]

coordenadas_lista = [
    (10.95749, -74.89345),  # VIA 11 CON CARRERA 8
    (11.00320, -74.81055),  # CARRERA 51B CON CALLE 79
    (10.94276, -74.78356),  # CALLE 30 CON CARRERA 6B
    (11.01742, -74.83438),  # CARRERA 53 ENTRE CALLE 104 Y 106
    (10.92959, -74.79918),  # CALLE 45 CON CARRERA 1
    (10.94667, -74.78458),  # CALLE 30 CON CARRERA 8
    (11.00457, -74.81435),  # CALLE 82 CON CARRERA 51B
    (11.01310, -74.83406),  # CARRERA 51B CON CALLE 103
    (10.94750, -74.81622),  # CARRERA 6 CON CALLE 72
    (10.95670, -74.83610),  # AVENIDA CIRCUNVALAR CON CARRERA 9G
    (11.01211, -74.79250),  # VIA 40 CON CALLE 73
    (10.96134, -74.79337),  # CALLE 45 CON CARRERA 20
    (11.00740, -74.81254),  # CALLE 82 CON CARRERA 56
    (10.95467, -74.79734),  # CALLE 45B CON CARRERA 14
    (10.96212, -74.79308),  # CALLE 45 CON CARRERA 21
    (10.95356, -74.77608),  # CARRERA 15 CON CALLE 21
    (10.99261, -74.80666),  # CALLE 72 CON CARRERA 44
    (11.01113, -74.81144),  # CALLE 84 ENTRE CARRERA 59 Y 59B
    (11.01561, -74.82071),  # CALLE 94 CON CARRERA 58
    (11.00788, -74.83351),  # CARRERA 46 CON CALLE 100
    (10.95849, -74.80349),  # CALLE 56 CON CARRERA 14
    (10.94252, -74.77770),  # CALLE 19 CON CARRERA 4C
    (10.98771, -74.78975),  # CALLE 53 CON CARRERA 45
    (10.97017, -74.83003),  # CALLE 87 CON CARRERA 21
    (10.98943, -74.81432),  # CALLE 76 CON CARRERA 38C-100
    (10.94020, -74.77858),  # CALLE 19 CON CARRERA 3D
    (11.00938, -74.81841),  # CARRERA 53 CON CALLE 86
    (10.99322, -74.80340),  # CALLE 70 CON CARRERA 46
    (10.98383, -74.77750),  # CALLE 34 CON CARRERA 45
    (10.97849, -74.83592),  # AVENIDA CIRCUNVALAR CON CARRERA 31
    (11.01575, -74.82514),  # CALLE 98 CON CARRERA 56
    (10.98087, -74.80040),  # CALLE 61 CON CARRERA 35
    (10.97925, -74.82361)   # CARRERA 27 CON CALLE 82C
]

coordenadas_dict = dict(zip(direcciones, coordenadas_lista))

## Visualización Geoespacial de Cámaras de Fotodetección en Barranquilla

Se construye un mapa interactivo que representa la ubicación geográfica de las cámaras de comparendos electrónicos en el municipio de Barranquilla, utilizando la capa base del shapefile municipal para contextualizar la distribución espacial. Para cada cámara georreferenciada, se calcula el total de comparendos (suma ponderada de `CANTIDAD_INFRACCIONES`), los volúmenes por código de infracción, y las fechas de primer y último registro. Los puntos se representan mediante marcadores circulares cuyo tamaño (radio) varía proporcionalmente al volumen de infracciones detectadas por cada cámara, permitiendo identificar visualmente los puntos de control con mayor actividad. Las cinco cámaras con mayor número de comparendos se distinguen con un color verde, mientras que el resto se muestran en color púrpura, facilitando la identificación de los puntos críticos. Al hacer clic en cada marcador, se despliega un popup con información detallada: dirección, total de comparendos, período de operación y desglose de infracciones por código. El mapa incluye el contorno del municipio de Barranquilla, una capa de control para activar/desactivar la visualización de las cámaras, una herramienta de medición de distancias y una leyenda fija que muestra el ranking de las cinco cámaras con mayor actividad. Esta visualización permite analizar la cobertura espacial del sistema de vigilancia, identificar zonas con mayor concentración de cámaras, distinguir los puntos de control más críticos y evaluar la relación entre la ubicación geográfica y el volumen de infracciones detectadas.

In [9]:
datos_camaras = []

for direccion in direcciones:
    df_camara = df_comparendos_electronicos[df_comparendos_electronicos['Camara_y_direccion'] == direccion]
    
    total_comparendos = df_camara['CANTIDAD_INFRACCIONES'].sum()
    comparendos_por_codigo = df_camara.groupby('COD_INFRACCION')['CANTIDAD_INFRACCIONES'].sum().to_dict()
    primera_fecha = df_camara['fecha_comparendo'].min().strftime('%Y-%m-%d')
    ultima_fecha = df_camara['fecha_comparendo'].max().strftime('%Y-%m-%d')
    
    datos_camaras.append({
        'direccion': direccion,
        'total': total_comparendos,
        'comparendos_por_codigo': comparendos_por_codigo,
        'primera_fecha': primera_fecha,
        'ultima_fecha': ultima_fecha,
        'coordenadas': coordenadas_dict[direccion]
    })
    
min_total = min(d['total'] for d in datos_camaras)
max_total = max(d['total'] for d in datos_camaras)
min_radius, max_radius = 5, 35

top_5_camaras = sorted(datos_camaras, key=lambda x: x['total'], reverse=True)[:5]
top_5_direcciones = {d['direccion'] for d in top_5_camaras}

for dato in datos_camaras:
    if max_total > min_total:
        dato['radius'] = min_radius + (dato['total'] - min_total) / (max_total - min_total) * (max_radius - min_radius)
    else:
        dato['radius'] = min_radius
        
mapa = folium.Map(location=[10.96854, -74.78132], zoom_start=12, tiles='cartodbpositron')

shapefile_path = "C:/Users/david/Documents/seminario_investigativo/MGN2025_MPIO_GRAFICO/MGN_ADM_MPIO_GRAFICO.shp"
gdf = gpd.read_file(shapefile_path)
gdf_barranquilla = gdf[gdf['mpio_cnmbr'] == 'BARRANQUILLA']

folium.GeoJson(
    gdf_barranquilla,
    style_function=lambda feature: {
        'fillColor': 'cornflowerblue',
        'color': 'black',
        'weight': 3,
        'fillOpacity': 0.5,
    },
    name='Municipio de Barranquilla'
).add_to(mapa)

def crear_popup_html(datos):
    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 14px; min-width: 200px;">
        <b style="font-size: 14px;">{datos['direccion']}</b><br>
        <hr style="margin: 5px 0;">
        <b>Total de comparendos:</b> {datos['total']:,}<br>
        <b>Primera fecha:</b> {datos['primera_fecha']}<br>
        <b>Última fecha:</b> {datos['ultima_fecha']}<br>
        <hr style="margin: 5px 0;">
        <b>Comparendos por codigo:</b><br>
    """
    
    for codigo, cantidad in datos['comparendos_por_codigo'].items():
        html += f"<b>{codigo}</b>: {int(cantidad):,}<br>"
    
    html += "</div>"
    return html

fg = folium.FeatureGroup(name="Camaras de Fotodeteccion")

for dato in datos_camaras:
    lat, lon = dato['coordenadas']
    popup_html = crear_popup_html(dato)
    
    if dato['direccion'] in top_5_direcciones:
        color = 'green'
        fill_color = 'green'
    else:
        color = 'mediumpurple'
        fill_color = 'mediumpurple'
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=dato['radius'],
        color=color,
        weight=2,
        fill=True,
        fill_color=fill_color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=folium.Tooltip(
            text=f"<span style='font-size: 14px;'><b>{dato['direccion']}</b></span><br><span style='font-size: 13px;'><b>Comparendos:</b> {dato['total']:,}</span>",
        )
    ).add_to(fg)
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color=color,
        weight=2,
        fill=True,
        fill_color=fill_color,
        fill_opacity=1.0,
        popup=folium.Popup(popup_html, max_width=300),  
        tooltip=folium.Tooltip(
            text=f"<span style='font-size: 14px;'><b>{dato['direccion']}</b></span><br><span style='font-size: 13px;'><b>Comparendos:</b> {dato['total']:,}</span>",
        )
    ).add_to(fg)

fg.add_to(mapa)

legend_html = '''
<div style="position: fixed; 
            bottom: 20px; right: 20px; 
            border:1px solid grey; 
            z-index:9999; 
            font-size:10px;
            background-color: white;
            padding: 6px;
            border-radius: 3px;
            box-shadow: 0 0 10px rgba(0,0,0,0.15);
            font-family: Arial, sans-serif;
            max-height: 250px;
            overflow-y: auto;">
    <b style="font-size: 11px;">Top 5 Cámaras con Más Comparendos</b><br>
    <hr style="margin: 3px 0;">
'''

for i, camara in enumerate(top_5_camaras, 1):
    legend_html += f'''
    <div style="margin-bottom: 2px;">
        <div style="display: inline-block; width: 10px; height: 10px; background-color: green; border: 1px solid #999; margin-right: 5px;"></div>
        <span><b>{i}. {camara['direccion']}:</b> {camara['total']:,}</span>
    </div>'''

legend_html += '''
    <hr style="margin: 3px 0;">
    <div style="margin-bottom: 2px;">
        <div style="display: inline-block; width: 10px; height: 10px; background-color: mediumpurple; border: 1px solid #999; margin-right: 5px;"></div>
        <span><b>Otras cámaras</b></span>
    </div>
</div>
'''

mapa.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(mapa)
folium.plugins.MeasureControl().add_to(mapa)

display(mapa)

## Visualización de la Distribución de Comparendos Electrónicos por Localidad de Barranquilla

Se realiza un análisis geoespacial que asigna cada cámara de fotodetección a una localidad del municipio de Barranquilla, utilizando un shapefile de polígonos de localidades y aplicando una unión espacial (spatial join) para determinar en qué polígono se encuentra cada punto. Para aquellas cámaras que no coinciden directamente con ningún polígono, se asigna la localidad más cercana mediante el cálculo de distancias mínimas a los polígonos de localidades. Con esta información, se construye un mapa interactivo donde cada localidad se representa con un polígono coloreado según el volumen total de comparendos (suma ponderada de `CANTIDAD_INFRACCIONES`) de todas las cámaras ubicadas en ella, utilizando una escala de colores azules que va del más claro (menor volumen) al más oscuro (mayor volumen). Al hacer clic en cada polígono de localidad, se despliega un popup con información detallada: nombre de la localidad, total de comparendos, cantidad de cámaras presentes y desglose de infracciones por código. Además, se incluye una capa con marcadores circulares que representan cada cámara individualmente, donde la cámara con mayor número de comparendos dentro de cada localidad se distingue en color verde, mientras que el resto se muestran en color púrpura. Una leyenda fija en la esquina inferior derecha resume el total de comparendos por localidad ordenado de mayor a menor, junto con la distinción de los marcadores. Este análisis permite identificar qué localidades concentran la mayor actividad de detección de infracciones, evaluar la distribución geográfica del sistema de cámaras, detectar los puntos de control más críticos dentro de cada localidad y determinar posibles áreas de mayor incidencia de ciertos tipos de infracción según la localidad.

In [10]:
geojson_path = "C:/Users/david/Documents/seminario_investigativo/localidades.geojson"
gdf_localidades = gpd.read_file(geojson_path)
gdf_localidades = gdf_localidades.to_crs('EPSG:4326')

geometrias_camaras = [Point(lon, lat) for lat, lon in [d['coordenadas'] for d in datos_camaras]]
gdf_camaras = gpd.GeoDataFrame(datos_camaras, geometry=geometrias_camaras, crs='EPSG:4326')

gdf_camaras_con_localidad = gpd.sjoin(gdf_camaras, gdf_localidades, how='left', predicate='within')

camaras_sin_localidad = gdf_camaras_con_localidad[gdf_camaras_con_localidad['index_right'].isna()]
if len(camaras_sin_localidad) > 0:
    camaras_sin_localidad_indices = camaras_sin_localidad.index
    for idx in camaras_sin_localidad_indices:
        punto = gdf_camaras_con_localidad.loc[idx, 'geometry']
        distancias = [punto.distance(localidad) for localidad in gdf_localidades['geometry']]
        localidad_cercana_idx = np.argmin(distancias)
        gdf_camaras_con_localidad.loc[idx, 'index_right'] = localidad_cercana_idx
        gdf_camaras_con_localidad.loc[idx, 'Localidad'] = gdf_localidades.iloc[localidad_cercana_idx]['Localidad']

for idx, row in gdf_camaras_con_localidad.iterrows():
    if 'Localidad' in row and pd.notna(row['Localidad']):
        datos_camaras[idx]['localidad_asignada'] = row['Localidad']
    else:
        punto = row['geometry']
        distancias = [punto.distance(localidad) for localidad in gdf_localidades['geometry']]
        localidad_cercana_idx = np.argmin(distancias)
        datos_camaras[idx]['localidad_asignada'] = gdf_localidades.iloc[localidad_cercana_idx]['Localidad']

camara_top_por_localidad = {}
for dato in datos_camaras:
    localidad = dato['localidad_asignada']
    if localidad not in camara_top_por_localidad:
        camara_top_por_localidad[localidad] = dato
    else:
        if dato['total'] > camara_top_por_localidad[localidad]['total']:
            camara_top_por_localidad[localidad] = dato

direcciones_top = {dato['direccion'] for dato in camara_top_por_localidad.values()}

comparendos_por_localidad = {}
comparendos_detalle_por_localidad = {}
camaras_por_localidad = {}

for dato in datos_camaras:
    localidad = dato['localidad_asignada']
    if localidad not in comparendos_por_localidad:
        comparendos_por_localidad[localidad] = 0
        comparendos_detalle_por_localidad[localidad] = {}
        camaras_por_localidad[localidad] = 0
    
    comparendos_por_localidad[localidad] += dato['total']
    camaras_por_localidad[localidad] += 1
    
    for codigo, cantidad in dato['comparendos_por_codigo'].items():
        if codigo not in comparendos_detalle_por_localidad[localidad]:
            comparendos_detalle_por_localidad[localidad][codigo] = 0
        comparendos_detalle_por_localidad[localidad][codigo] += cantidad

localidades_ordenadas = sorted(comparendos_por_localidad.items(), key=lambda x: x[1])
n_localidades = len(comparendos_por_localidad)

blues_palette = sns.color_palette("Blues", n_colors=n_localidades)

colores_localidades = {}
for i, (localidad, _) in enumerate(localidades_ordenadas):
    r, g, b = blues_palette[i]
    colores_localidades[localidad] = '#%02x%02x%02x' % (int(r*255), int(g*255), int(b*255))

color_mas_claro = '#%02x%02x%02x' % tuple(int(c*255) for c in blues_palette[0])
color_mas_oscuro = '#%02x%02x%02x' % tuple(int(c*255) for c in blues_palette[-1])

mapa_localidades = folium.Map(location=[10.96854, -74.78132], zoom_start=12, tiles='cartodbpositron')

def crear_popup_localidad(localidad, total, detalle_codigos, num_camaras):
    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 14px; min-width: 200px;">
        <b style="font-size: 14px;">{localidad}</b><br>
        <hr style="margin: 5px 0;">
        <b>Total de comparendos:</b> {total:,}<br>
        <b>Cantidad de cámaras:</b> {num_camaras}<br>
        <hr style="margin: 5px 0;">
        <b>Comparendos por código:</b><br>
    """
    
    for codigo, cantidad in sorted(detalle_codigos.items()):
        html += f"<b>{codigo}</b>: {int(cantidad):,}<br>"
    
    html += "</div>"
    return folium.Popup(html, max_width=300)

def tooltip_localidad(localidad, total):
    return folium.Tooltip(
        text=f"<span style='font-size: 14px;'><b>{localidad}</b></span><br><span style='font-size: 13px;'><b>Comparendos:</b> {total:,}</span>",
        sticky=True
    )

for idx, row in gdf_localidades.iterrows():
    localidad = row['Localidad']
    if localidad in comparendos_por_localidad:
        total = comparendos_por_localidad[localidad]
        color = colores_localidades[localidad]
        detalle = comparendos_detalle_por_localidad[localidad]
        num_camaras = camaras_por_localidad[localidad]
        
        folium.GeoJson(
            row['geometry'],
            style_function=lambda feature, c=color: {
                'fillColor': c,
                'color': 'black',
                'weight': 2.5,
                'fillOpacity': 0.5,
            },
            popup=crear_popup_localidad(localidad, total, detalle, num_camaras),
            tooltip=tooltip_localidad(localidad, total),
            name=f"Localidad: {localidad}",
            overlay=True,
            control=True
        ).add_to(mapa_localidades)

fg_camaras = folium.FeatureGroup(name="Cámaras de Fotodetección", overlay=True, control=True)

for dato in datos_camaras:
    lat, lon = dato['coordenadas']
    popup_html = crear_popup_html(dato)
    
    if dato['direccion'] in direcciones_top:
        color = 'green'
        fill_color = 'green'
    else:
        color = 'mediumpurple'
        fill_color = 'mediumpurple'
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color=color,
        weight=2,
        fill=True,
        fill_color=fill_color,
        fill_opacity=1.0,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=folium.Tooltip(
            text=f"<span style='font-size: 14px;'><b>{dato['direccion']}</b></span><br><span style='font-size: 13px;'><b>Comparendos:</b> {dato['total']:,}</span>",
        )
    ).add_to(fg_camaras)

fg_camaras.add_to(mapa_localidades)

min_total_loc = min(comparendos_por_localidad.values())
max_total_loc = max(comparendos_por_localidad.values())

colormap = cm.LinearColormap(
    colors=[color_mas_claro, color_mas_oscuro],
    vmin=min_total_loc,
    vmax=max_total_loc,
    caption='Comparendos por Localidad'
)

colormap.add_to(mapa_localidades)

legend_html = '''
<div style="position: fixed; 
            bottom: 20px; right: 20px; 
            border:1px solid grey; 
            z-index:9999; 
            font-size:10px;
            background-color: white;
            padding: 6px;
            border-radius: 3px;
            box-shadow: 0 0 10px rgba(0,0,0,0.15);
            font-family: Arial, sans-serif;
            max-height: 250px;
            overflow-y: auto;">
    <b style="font-size: 11px;">Total de Comparendos por Localidad</b><br>
    <hr style="margin: 3px 0;">
'''

for localidad, total in sorted(comparendos_por_localidad.items(), key=lambda x: x[1], reverse=True):
    color = colores_localidades[localidad]
    legend_html += f'''
    <div style="margin-bottom: 2px;">
        <div style="display: inline-block; width: 10px; height: 10px; background-color: {color}; border: 1px solid #999; margin-right: 5px;"></div>
        <span><b>{localidad}:</b> {total:,}</span>
    </div>
    '''

legend_html += '''
    <hr style="margin: 3px 0;">
    <div style="margin-bottom: 2px;">
        <div style="display: inline-block; width: 10px; height: 10px; background-color: green; border: 1px solid #999; margin-right: 5px;"></div>
        <span><b>Cámara top por localidad</b></span>
    </div>
    <div style="margin-bottom: 2px;">
        <div style="display: inline-block; width: 10px; height: 10px; background-color: mediumpurple; border: 1px solid #999; margin-right: 5px;"></div>
        <span><b>Otras cámaras</b></span>
    </div>
</div>
'''

mapa_localidades.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(mapa_localidades)
folium.plugins.MeasureControl().add_to(mapa_localidades)

display(mapa_localidades)